In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load the cleaned historical dataset and sort by date
print("Loading baseline dataset...")
df = pd.read_csv('../data/processed/cleaned_historical_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

Loading baseline dataset...


In [3]:
# apply tournament weights to the dataset
print("Applying tournament weights...")

def get_tournament_weight(tournament):
    # Convert to string and lowercase to avoid case-sensitivity or NaN bugs
    t_lower = str(tournament).lower()
    
    # 1. Handle Qualifiers first so they don't accidentally match the main tournament checks
    if 'qualification' in t_lower or 'qualifier' in t_lower:
        return 2.0
        
    # 2. Main World Cup tournament
    elif 'world cup' in t_lower:
        return 4.0
        
    # 3. Major Continental Tournaments
    elif any(comp in t_lower for comp in ['continental', 'copa america', 'copa américa', 'euro', 'africa cup', 'afcon', 'asian cup', 'gold cup']):
        return 3.0
        
    # 4. Friendlies and minor tournaments
    else:
        return 1.0

df['match_weight'] = df['tournament'].apply(get_tournament_weight)

Applying tournament weights...


In [4]:
print("Pivoting to long format to calculate sequential team form...")

# Create Home Perspective
home_df = df[['date', 'home_team', 'home_score', 'away_score', 'match_weight']].copy()
home_df.columns = ['date', 'team', 'goals_for', 'goals_against', 'weight']
home_df['points'] = np.where(home_df['goals_for'] > home_df['goals_against'], 3, 
                    np.where(home_df['goals_for'] == home_df['goals_against'], 1, 0))

# Create Away Perspective
away_df = df[['date', 'away_team', 'away_score', 'home_score', 'match_weight']].copy()
away_df.columns = ['date', 'team', 'goals_for', 'goals_against', 'weight']
away_df['points'] = np.where(away_df['goals_for'] > away_df['goals_against'], 3, 
                    np.where(away_df['goals_for'] == away_df['goals_against'], 1, 0))

# Combine and sort strictly by timeline
long_df = pd.concat([home_df, away_df]).sort_values(['team', 'date']).reset_index(drop=True)

# Apply Weighting to Points
long_df['weighted_points'] = long_df['points'] * long_df['weight']

Pivoting to long format to calculate sequential team form...


In [5]:
# Calculate last 5 matches form strictly BEFORE the current match
long_df['recent_form'] = long_df.groupby('team')['weighted_points'].transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
long_df['recent_scoring_form'] = long_df.groupby('team')['goals_for'].transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())

# Isolate the features back into a lookup table
form_lookup = long_df[['date', 'team', 'recent_form', 'recent_scoring_form']]

# Merge back into the wide dataset
df = pd.merge(df, form_lookup, left_on=['date', 'home_team'], right_on=['date', 'team'], how='left')
df = df.rename(columns={'recent_form': 'home_recent_form', 'recent_scoring_form': 'home_recent_scoring'})
df = df.drop(columns=['team'])

df = pd.merge(df, form_lookup, left_on=['date', 'away_team'], right_on=['date', 'team'], how='left')
df = df.rename(columns={'recent_form': 'away_recent_form', 'recent_scoring_form': 'away_recent_scoring'})
df = df.drop(columns=['team'])

In [6]:
print("Calculating perspective-accurate historical Head-to-Head metrics...")

# 1. Establish static chronological anchors alphabetically
df['team_1'] = df.apply(lambda r: min(r['home_team'], r['away_team']), axis=1)
df['team_2'] = df.apply(lambda r: max(r['home_team'], r['away_team']), axis=1)

# 2. Calculate historical goal difference ALWAYS from team_1's perspective
df['team_1_gd'] = np.where(
    df['home_team'] == df['team_1'],
    df['home_score'] - df['away_score'], # team_1 was home
    df['away_score'] - df['home_score']  # team_1 was away
)

# 3. Group by the anchor pair, shift to prevent leakage, and calculate expanding historical average
df['anchor_h2h_gd'] = df.groupby(['team_1', 'team_2'])['team_1_gd'].transform(
    lambda x: x.shift(1).expanding().mean()
)
df['anchor_h2h_gd'] = df['anchor_h2h_gd'].fillna(0)

# 4. Resolve the metric back to the CURRENT match home team's perspective
# If current home_team is team_1, keep the sign. If current home_team is team_2, invert it!
df['h2h_goal_diff'] = np.where(
    df['home_team'] == df['team_1'],
    df['anchor_h2h_gd'],
    -df['anchor_h2h_gd']
)

# Clean up our temporary anchor columns
df = df.drop(columns=['team_1', 'team_2', 'team_1_gd', 'anchor_h2h_gd'])

Calculating perspective-accurate historical Head-to-Head metrics...


In [7]:
# Drop matches from before 1993 (when FIFA rankings started) as they will have NaNs for ranks
df_final = df.dropna(subset=['home_fifa_rank', 'away_fifa_rank']).copy()

# Fill initial NaN form values for a team's very first 5 matches with a neutral 0
df_final[['home_recent_form', 'away_recent_form', 'home_recent_scoring', 'away_recent_scoring']] = df_final[['home_recent_form', 'away_recent_form', 'home_recent_scoring', 'away_recent_scoring']].fillna(0)

df_final.to_csv('../data/processed/training_features.csv', index=False)
print("Phase 2 Complete! Fully engineered feature matrix saved to '../data/processed/training_features.csv'")
print(df_final[['date', 'home_team', 'away_team', 'home_recent_form', 'h2h_goal_diff']].tail())

Phase 2 Complete! Fully engineered feature matrix saved to '../data/processed/training_features.csv'
            date home_team  away_team  home_recent_form  h2h_goal_diff
49742 2026-06-27    Panama    England               1.2           -5.0
49743 2026-06-27   Algeria    Austria               1.4           -2.0
49744 2026-06-27    Jordan  Argentina               0.4           -0.0
49745 2026-06-27  Colombia   Portugal               0.6            0.0
49746 2026-06-27   Croatia      Ghana               0.6            0.0
